In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Read CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes where present
for df in (metric, trace, log, error_logs):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to build aggregated summary (count, mean, min, max, 90th, 95th) grouped by two columns
def agg_summary(df, group_cols, value_col='value'):
    g = df.groupby(group_cols)[value_col]
    agg = g.agg(
        count='count',
        mean='mean',
        min='min',
        max='max',
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95),
    ).reset_index()
    # Round numeric columns for compact display
    num_cols = ['mean', 'min', 'max', 'p90', 'p95']
    for c in num_cols:
        if c in agg.columns:
            agg[c] = agg[c].round(6)
    agg = agg.sort_values('count', ascending=False).reset_index(drop=True)
    return agg

# 1) metric.csv summary
metric_summary = agg_summary(metric, ['cmdb_id', 'kpi_name'], 'value').head(50)
metric_cmdbs = pd.Series(sorted(metric['cmdb_id'].dropna().unique()), name='distinct_cmdb_id')
metric_kpis = pd.Series(sorted(metric['kpi_name'].dropna().unique()), name='distinct_kpi_name')

# 2) trace.csv summary
trace_summary = agg_summary(trace, ['cmdb_id', 'trace_name'], 'value').head(50)
trace_cmdbs = pd.Series(sorted(trace['cmdb_id'].dropna().unique()), name='distinct_cmdb_id')
trace_names = pd.Series(sorted(trace['trace_name'].dropna().unique()), name='distinct_trace_name')

# 3) log.csv summary
log_summary = agg_summary(log, ['cmdb_id', 'log_name'], 'value').head(50)
log_cmdbs = pd.Series(sorted(log['cmdb_id'].dropna().unique()), name='distinct_cmdb_id')
log_names = pd.Series(sorted(log['log_name'].dropna().unique()), name='distinct_log_name')

# 4) error_logs.csv summary: count, distinct cmdb_id values, earliest and latest timestamp
error_count = len(error_logs)
error_distinct_cmdbs = pd.Series(sorted(error_logs['cmdb_id'].dropna().unique()), name='distinct_cmdb_id')
if 'timestamp' in error_logs.columns and not error_logs['timestamp'].isna().all():
    error_earliest = error_logs['timestamp'].min()
    error_latest = error_logs['timestamp'].max()
else:
    error_earliest = pd.NaT
    error_latest = pd.NaT

error_summary = pd.DataFrame({
    'row_count': [error_count],
    'earliest_timestamp_utc': [error_earliest],
    'latest_timestamp_utc': [error_latest],
})

# Keep outputs compact: show aggregated top rows and the distinct identifier lists and error summary
metric_summary, metric_cmdbs, metric_kpis, trace_summary, trace_cmdbs, trace_names, log_summary, log_cmdbs, log_names, error_summary, error_distinct_cmdbs

```
Out[1]:
```
metric_summary.head(5).to_csv(index=False), \
f"metric_distinct_cmdb_count: {metric_cmdbs.shape[0]}, metric_distinct_kpi_count: {metric_kpis.shape[0]}", \
"metric_top_kpis_csv_preview:\n" + metric_summary.head(5)[['cmdb_id','kpi_name','count','mean','min','max','p90','p95']].to_csv(index=False), \
trace_summary.head(5).to_csv(index=False), \
f"trace_distinct_cmdb_count: {trace_cmdbs.shape[0]}, trace_distinct_trace_name_count: {trace_names.shape[0]}", \
"trace_top_csv_preview:\n" + trace_summary.head(5)[['cmdb_id','trace_name','count','mean','min','max','p90','p95']].to_csv(index=False), \
log_summary.head(5).to_csv(index=False), \
f"log_distinct_cmdb_count: {log_cmdbs.shape[0]}, log_distinct_log_name_count: {log_names.shape[0]}", \
"log_top_csv_preview:\n" + log_summary.head(5)[['cmdb_id','log_name','count','mean','min','max','p90','p95']].to_csv(index=False), \
error_summary.to_csv(index=False), \
f"error_distinct_cmdbs: {', '.join(error_distinct_cmdbs.tolist()) if len(error_distinct_cmdbs)>0 else '[]'}"

The original code execution output of IPython Kernel is also provided below for reference:

(                        cmdb_id    kpi_name  count          mean           min           max           p90           p95
0            ts-voucher-service      socket     25      4.000000      4.000000      4.000000      4.000000      4.000000
1   ts-admin-basic-info-service         cpu     25      0.537918      0.387032      0.789384      0.662118      0.739390
2   ts-admin-basic-info-service      diskio     25  52134.237994  38098.818738  64984.927812  61811.782714  62996.948310
3   ts-admin-basic-info-service       error     25      0.000000      0.000000      0.000000      0.000000      0.000000
4   ts-admin-basic-info-service  latency-50     25      0.017489      0.016724      0.017989      0.017742      0.017914
..                          ...         ...    ...           ...           ...           ...           ...           ...
45              ts-config-mongo      socket     25      7.000000      7.000000      7.000000      7.000000      7.000000
46            ts-config-service         cpu     25      1.490213      1.149884      3.398252      1.783985      2.397355
47            ts-config-service      diskio     25  51908.555988  38487.710961  65208.652564  60030.429379  61880.365676
48            ts-config-service  latency-50     25      0.003722      0.003453      0.004086      0.003870      0.003893
49             ts-basic-service  latency-50     25      0.008178      0.008056      0.008336      0.008323      0.008328

[50 rows x 8 columns], 0      ts-admin-basic-info-service
1           ts-admin-order-service
2           ts-admin-route-service
3          ts-admin-travel-service
4            ts-admin-user-service
                  ...             
63                   ts-user-mongo
64                 ts-user-service
65    ts-verification-code-service
66                ts-voucher-mysql
67              ts-voucher-service
Name: distinct_cmdb_id, Length: 68, dtype: object, 0           cpu
1        diskio
2         error
3    latency-50
4    latency-90
5           mem
6        socket
7      workload
Name: distinct_kpi_name, dtype: object,               cmdb_id                                  trace_name  count        mean       min         max         p90         p95
0   ts-travel-service     trace.from_ts-travel-service.error_rate     25    0.000000  0.000000    0.000000    0.000000    0.000000
1   ts-travel-service   trace.from_ts-travel-service.duration_p95     25    0.073358  0.046133    0.176277    0.099966    0.106190
2   ts-travel-service  trace.from_ts-travel-service.duration_mean     25    0.020535  0.009398    0.036928    0.032200    0.033788
3   ts-travel-service          trace.to_ts-seat-service.row_count     25   87.440000  2.000000  116.000000  109.600000  112.000000
4   ts-travel-service         trace.to_ts-seat-service.error_rate     25    0.000000  0.000000    0.000000    0.000000    0.000000
..                ...                                         ...    ...         ...       ...         ...         ...         ...
45   ts-price-service         trace.to_ts-price-service.row_count     25  263.120000  6.000000  318.000000  300.000000  308.800000
46   ts-route-service   trace.from_ts-basic-service.duration_mean     25    0.023726  0.002885    0.116754    0.042839    0.044915
47   ts-route-service    trace.from_ts-basic-service.duration_p95     25    0.051015  0.003330    0.316046    0.077458    0.106040
48   ts-route-service      trace.from_ts-basic-service.error_rate     25    0.000000  0.000000    0.000000    0.000000    0.000000
49  ts-config-service        trace.from_ts-seat-service.row_count     25  233.200000  6.000000  278.000000  265.600000  274.000000

[50 rows x 8 columns], 0                            root
1     ts-admin-basic-info-service
2         ts-admin-travel-service
3            ts-assurance-service
4                 ts-auth-service
                 ...             
23          ts-ticketinfo-service
24               ts-train-service
25              ts-travel-service
26             ts-travel2-service
27                ts-user-service
Name: distinct_cmdb_id, Length: 28, dtype: object, 0                          trace.from_root.duration_mean
1                           trace.from_root.duration_p95
2                             trace.from_root.error_rate
3                              trace.from_root.row_count
4      trace.from_ts-admin-basic-info-service.duratio...
                             ...                        
215                trace.to_ts-travel2-service.row_count
216               trace.to_ts-user-service.duration_mean
217                trace.to_ts-user-service.duration_p95
218                  trace.to_ts-user-service.error_rate
219                   trace.to_ts-user-service.row_count
Name: distinct_trace_name, Length: 220, dtype: object,                   cmdb_id         log_name  count         mean  min   max     p90     p95
0         ts-auth-service    log.row_count     25   309.280000   12   350   338.4   340.0
1         ts-auth-service  log.error_count     25     0.000000    0     0     0.0     0.0
2        ts-basic-service    log.row_count     25  2241.840000   63  2703  2549.0  2637.0
3        ts-basic-service  log.error_count     25     0.000000    0     0     0.0     0.0
4         ts-ui-dashboard    log.row_count     25   256.160000    5   291   284.6   289.4
..                    ...              ...    ...          ...  ...   ...     ...     ...
45  ts-notification-mongo  log.error_count     24     0.000000    0     0     0.0     0.0
46      ts-food-map-mongo  log.error_count     24     0.000000    0     0     0.0     0.0
47      ts-food-map-mongo    log.row_count     24     1.000000    1     1     1.0     1.0
48    ts-food-map-service  log.error_count     24     0.000000    0     0     0.0     0.0
49    ts-food-map-service    log.row_count     24    20.416667   11    29    25.0    28.4

[50 rows x 8 columns], 0     ts-admin-basic-info-service
1         ts-admin-travel-service
2              ts-assurance-mongo
3            ts-assurance-service
4                   ts-auth-mongo
                 ...             
40              ts-travel-service
41               ts-travel2-mongo
42             ts-travel2-service
43                ts-ui-dashboard
44                  ts-user-mongo
Name: distinct_cmdb_id, Length: 45, dtype: object, 0    log.error_count
1      log.row_count
Name: distinct_log_name, dtype: object,    row_count earliest_timestamp_utc latest_timestamp_utc
0          0                    NaT                  NaT, Series([], Name: distinct_cmdb_id, dtype: object))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing dataframes: metric, trace, log (timestamps already parsed as UTC)
candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# Incident window in UTC
start = pd.to_datetime('2024-01-24 07:53:02', utc=True)
end   = pd.to_datetime('2024-01-24 08:23:02', utc=True)

# Helper to compute global P95 per group (on full series) and find anomalies in window
def find_anomalies(df, group_cols, value_col='value'):
    # global p95 using full series
    p95_df = df.groupby(group_cols)[value_col].quantile(0.95).reset_index(name='global_p95')
    # limit to candidate services
    p95_df = p95_df[p95_df['cmdb_id'].isin(candidates)]
    if p95_df.empty:
        # return empty table with desired columns
        cols = group_cols + ['anomaly_count_in_window','earliest_anomaly_timestamp_utc','max_value_in_window','global_p95']
        return pd.DataFrame(columns=cols)
    # filter original df to candidates and incident window
    window_df = df[df['cmdb_id'].isin(candidates) & df['timestamp'].between(start, end)]
    if window_df.empty:
        cols = group_cols + ['anomaly_count_in_window','earliest_anomaly_timestamp_utc','max_value_in_window','global_p95']
        return pd.DataFrame(columns=cols)
    # merge window rows with global p95
    merged = window_df.merge(p95_df, on=group_cols, how='left')
    # keep only rows where value >= global_p95 (and global_p95 not null)
    anomalies = merged[merged[value_col] >= merged['global_p95']].copy()
    if anomalies.empty:
        cols = group_cols + ['anomaly_count_in_window','earliest_anomaly_timestamp_utc','max_value_in_window','global_p95']
        return pd.DataFrame(columns=cols)
    # aggregate anomalies per group
    agg = anomalies.groupby(group_cols).agg(
        anomaly_count_in_window = (value_col, 'count'),
        earliest_anomaly_timestamp_utc = ('timestamp', 'min'),
        max_value_in_window = (value_col, 'max'),
    ).reset_index()
    # attach global_p95
    agg = agg.merge(p95_df, on=group_cols, how='left')
    # rounding for compactness
    agg['global_p95'] = agg['global_p95'].round(6)
    agg['max_value_in_window'] = agg['max_value_in_window'].round(6)
    # sort as requested
    agg = agg.sort_values(['anomaly_count_in_window','max_value_in_window'], ascending=[False, False]).reset_index(drop=True)
    return agg

# 1) Metrics
metric_anomalies = find_anomalies(metric, ['cmdb_id','kpi_name'], 'value').head(20)

# 2) Traces
trace_anomalies = find_anomalies(trace, ['cmdb_id','trace_name'], 'value').head(20)

# 3) Logs
log_anomalies = find_anomalies(log, ['cmdb_id','log_name'], 'value').head(20)

# Display compact results
metric_anomalies, trace_anomalies, log_anomalies

```
Out[2]:
```
summary = (
"Summary of anomalies in the incident window (2024-01-24 07:53:02 to 2024-01-24 08:23:02 UTC):\n\n"
"1) Metrics (threshold = global P95 over full series):\n"
"- Many trivial 'error' / trace error_rate / log.error_count series have global P95 = 0, so all zero samples in the window are flagged (e.g. ts-auth-service error: 25 hits, p95=0). These are not meaningful alerts.\n"
"- Relevant resource spikes to investigate:\n"
"  * Memory: multiple services show P95 breaches (each with 2 anomalous minutes):\n"
"    - ts-route-service: mem (2 anomalies), max ~7.34e8, global P95 ~7.32e8\n"
"    - ts-travel-service: mem (2), max ~2.71e8, global P95 ~2.71e8\n"
"    - ts-order-service: mem (2), max ~2.68e8, global P95 ~2.67e8\n"
"    - ts-train-service: mem (2), max ~2.51e8, global P95 ~2.51e8\n"
"    - ts-auth-service: mem (2), max ~2.47e8, global P95 ~2.45e8\n"
"  * Disk I/O: several services breached P95 for diskio (each with 2 anomalous minutes), notable:\n"
"    - ts-route-service: diskio max ~4.53e7 (p95 ~4.36e7)\n"
"    - ts-order-service, ts-auth-service, ts-train-service, ts-travel-service also show diskio anomalies.\n"
"  * CPU / workload / socket: some candidate services show short spikes (2 minutes each) above P95 (e.g., ts-route cpu, workload, socket; ts-auth cpu/socket).\n\n"
"2) Traces (global P95 over full series, then window):\n"
"- Many trace 'error_rate' series have global P95 = 0 and therefore appear flagged (25 hits with value 0). These are not actionable.\n"- No clear elevated trace latency/duration anomalies surfaced in the top results for these candidates beyond the zero-error-rate artifacts.\n\n"
"3) Logs (global P95 over full series, then window):\n"
"- log.error_count: global P95 = 0 for these services -> flagged zeros (not helpful).\n"- log.row_count shows meaningful spikes above P95:\n"
"  * ts-auth-service: 3 anomalous minutes, max 350 (global P95 ~340)\n" 
"  * ts-travel-service: 2 anomalous minutes, max 1966 (global P95 ~1847.2)\n" 
"  * ts-route-service: 2 anomalous minutes, max 1091 (global P95 ~1075.2)\n" 
"  * ts-order-service: 2 anomalous minutes, max 446 (global P95 ~423.6)\n\n"
"Interpretation & next steps:\n"
"- Primary signals point to resource pressure (memory and disk I/O) across multiple candidate services (route, travel, order, train, auth) during the incident window. These resource anomalies coincide with elevated log throughput (log.row_count) for auth, travel, route, order.\n"
"- Many 'error' and trace error_rate/log.error_count series have P95 = 0 which causes unhelpful zero-valued alerts; ignore these for root-cause.\n"
"- Actionable follow-up: focus on memory and disk I/O for ts-route-service, ts-travel-service, ts-order-service, ts-train-service, ts-auth-service, and inspect the increased log traffic for ts-auth, ts-travel, ts-route, ts-order for related errors or traffic surges.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id  kpi_name  anomaly_count_in_window earliest_anomaly_timestamp_utc  max_value_in_window    global_p95
0     ts-auth-service     error                       25      2024-01-24 07:56:00+00:00         0.000000e+00  0.000000e+00
1    ts-route-service       mem                        2      2024-01-24 08:10:00+00:00         7.340003e+08  7.321495e+08
2   ts-travel-service       mem                        2      2024-01-24 08:19:00+00:00         2.713231e+08  2.712864e+08
3    ts-order-service       mem                        2      2024-01-24 08:13:00+00:00         2.676754e+08  2.668523e+08
4    ts-train-service       mem                        2      2024-01-24 08:18:00+00:00         2.508308e+08  2.508177e+08
5     ts-auth-service       mem                        2      2024-01-24 08:07:00+00:00         2.469997e+08  2.446292e+08
6    ts-route-service    diskio                        2      2024-01-24 08:13:00+00:00         4.533448e+07  4.358652e+07
7    ts-order-service    diskio                        2      2024-01-24 08:15:00+00:00         1.878737e+06  1.821038e+06
8     ts-auth-service    diskio                        2      2024-01-24 08:14:00+00:00         1.810451e+06  1.721153e+06
9    ts-train-service    diskio                        2      2024-01-24 08:09:00+00:00         6.228674e+04  6.090494e+04
10  ts-travel-service    diskio                        2      2024-01-24 08:05:00+00:00         6.095932e+04  5.738472e+04
11   ts-route-service       cpu                        2      2024-01-24 08:15:00+00:00         4.922756e+01  4.902310e+01
12   ts-route-service  workload                        2      2024-01-24 08:19:00+00:00         3.622233e+01  3.527090e+01
13   ts-train-service  workload                        2      2024-01-24 08:19:00+00:00         2.402233e+01  2.228449e+01
14   ts-route-service    socket                        2      2024-01-24 08:10:00+00:00         2.200000e+01  1.934000e+01
15  ts-travel-service    socket                        2      2024-01-24 08:12:00+00:00         2.106667e+01  1.949333e+01
16    ts-auth-service       cpu                        2      2024-01-24 08:02:00+00:00         1.773783e+01  1.687433e+01
17   ts-train-service    socket                        2      2024-01-24 08:12:00+00:00         1.683333e+01  1.570333e+01
18    ts-auth-service    socket                        2      2024-01-24 08:02:00+00:00         1.470000e+01  1.424333e+01
19   ts-order-service    socket                        2      2024-01-24 08:13:00+00:00         1.323333e+01  1.307000e+01,               cmdb_id                                trace_name  anomaly_count_in_window earliest_anomaly_timestamp_utc  max_value_in_window  global_p95
0     ts-auth-service                trace.from_root.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
1     ts-auth-service     trace.from_ts-auth-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
2     ts-auth-service       trace.to_ts-auth-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
3    ts-order-service    trace.from_ts-order-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
4    ts-order-service     trace.from_ts-seat-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
5    ts-order-service   trace.from_ts-travel-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
6    ts-order-service      trace.to_ts-order-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
7    ts-route-service    trace.from_ts-basic-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
8    ts-route-service    trace.from_ts-route-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
9    ts-route-service   trace.from_ts-travel-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
10   ts-route-service  trace.from_ts-travel2-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
11   ts-route-service      trace.to_ts-route-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
12   ts-train-service    trace.from_ts-basic-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
13   ts-train-service    trace.from_ts-train-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
14   ts-train-service   trace.from_ts-travel-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
15   ts-train-service  trace.from_ts-travel2-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
16   ts-train-service      trace.to_ts-train-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
17  ts-travel-service                trace.from_root.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
18  ts-travel-service     trace.from_ts-seat-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
19  ts-travel-service   trace.from_ts-travel-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0,              cmdb_id         log_name  anomaly_count_in_window earliest_anomaly_timestamp_utc  max_value_in_window  global_p95
0    ts-auth-service  log.error_count                       25      2024-01-24 07:56:00+00:00                    0         0.0
1   ts-order-service  log.error_count                       25      2024-01-24 07:56:00+00:00                    0         0.0
2   ts-route-service  log.error_count                       25      2024-01-24 07:56:00+00:00                    0         0.0
3  ts-travel-service  log.error_count                       25      2024-01-24 07:56:00+00:00                    0         0.0
4    ts-auth-service    log.row_count                        3      2024-01-24 07:57:00+00:00                  350       340.0
5  ts-travel-service    log.row_count                        2      2024-01-24 08:02:00+00:00                 1966      1847.2
6   ts-route-service    log.row_count                        2      2024-01-24 07:57:00+00:00                 1091      1075.2
7   ts-order-service    log.row_count                        2      2024-01-24 08:05:00+00:00                  446       423.6)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing dataframes: metric, log (timestamps already parsed as UTC)
candidates = ['ts-route-service','ts-travel-service','ts-order-service','ts-train-service','ts-auth-service']
metric_kpis = ['mem','diskio','cpu','workload','socket']
log_kpi = 'log.row_count'

# Incident window
start = pd.to_datetime('2024-01-24 07:53:02', utc=True)
end   = pd.to_datetime('2024-01-24 08:23:02', utc=True)

# Helper to compute groups for a single group's anomaly rows
def group_consecutive_anomalies(df_anom, time_col='timestamp', max_gap=pd.Timedelta(seconds=90)):
    if df_anom.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','kpi_name','group_start_timestamp_utc','group_end_timestamp_utc',
            'points_in_group','max_value_in_group','global_p95','percent_over'
        ])
    df = df_anom.sort_values(time_col).copy()
    df['gap'] = df[time_col].diff().gt(max_gap).fillna(True)
    df['grp'] = df['gap'].cumsum()
    agg = df.groupby('grp').agg(
        group_start_timestamp_utc = (time_col, 'min'),
        group_end_timestamp_utc   = (time_col, 'max'),
        points_in_group           = (time_col, 'count'),
        max_value_in_group        = ('value', 'max')
    ).reset_index(drop=True)
    return agg

# Compute global P95 and full-series max for metric KPIs
metric_full = metric[metric['cmdb_id'].isin(candidates) & metric['kpi_name'].isin(metric_kpis)].copy()
if metric_full.empty:
    metric_p95 = pd.DataFrame(columns=['cmdb_id','kpi_name','global_p95','full_max'])
else:
    metric_p95 = metric_full.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_p95=lambda x: x.quantile(0.95),
        full_max='max'
    ).reset_index()
    metric_p95['global_p95'] = metric_p95['global_p95'].round(6)

# Compute global P95 and full-series max for log.row_count
log_full = log[log['cmdb_id'].isin(candidates) & (log['log_name'] == log_kpi)].copy()
if log_full.empty:
    log_p95 = pd.DataFrame(columns=['cmdb_id','kpi_name','global_p95','full_max'])
else:
    log_p95 = log_full.groupby(['cmdb_id'])['value'].agg(
        global_p95=lambda x: x.quantile(0.95),
        full_max='max'
    ).reset_index().rename(columns={'cmdb_id':'cmdb_id'})
    log_p95['kpi_name'] = log_kpi
    log_p95 = log_p95[['cmdb_id','kpi_name','global_p95','full_max']]
    log_p95['global_p95'] = log_p95['global_p95'].round(6)

# Combine thresholds
thresholds = pd.concat([metric_p95, log_p95], ignore_index=True, sort=False)
# If thresholds empty, return empty result
if thresholds.empty:
    groups_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','group_start_timestamp_utc','group_end_timestamp_utc',
        'points_in_group','max_value_in_group','global_p95','percent_over'
    ])
else:
    # Prepare windowed data for metric and log
    metric_window = metric[(metric['timestamp'].between(start,end)) & metric['cmdb_id'].isin(candidates) & metric['kpi_name'].isin(metric_kpis)].copy()
    # align column name for kpi
    metric_window = metric_window[['cmdb_id','kpi_name','timestamp','value']]

    log_window = log[(log['timestamp'].between(start,end)) & log['cmdb_id'].isin(candidates) & (log['log_name']==log_kpi)].copy()
    if not log_window.empty:
        log_window = log_window[['cmdb_id']].assign(kpi_name=log_kpi)
        log_window = log_window.join(log_window.index.to_series().rename('idx'))  # keep index
    # actually construct log_window with appropriate columns
    if not log_window.empty:
        log_window = log[(log['timestamp'].between(start,end)) & log['cmdb_id'].isin(candidates) & (log['log_name']==log_kpi)].copy()
        log_window = log_window[['cmdb_id','timestamp','value']].rename(columns={'value':'value'})
        log_window['kpi_name'] = log_kpi
        log_window = log_window[['cmdb_id','kpi_name','timestamp','value']]

    # unify windowed rows
    window_all = pd.concat([metric_window, log_window], ignore_index=True, sort=False)
    # Iterate over each threshold group, find anomaly rows and group them
    groups_list = []
    for _, row in thresholds.iterrows():
        cmdb = row['cmdb_id']
        kpi = row['kpi_name']
        gp95 = float(row['global_p95'])
        fmax = float(row['full_max'])
        # If full series all zeros (full_max == 0) -> skip entirely
        if fmax == 0:
            continue
        # select window rows for this group
        rows = window_all[(window_all['cmdb_id']==cmdb) & (window_all['kpi_name']==kpi)].copy()
        if rows.empty:
            continue
        # determine anomaly condition:
        if gp95 > 0:
            anom = rows[rows['value'] >= gp95].copy()
        else:
            # gp95 == 0 but full_max > 0 -> consider value > 0 as anomaly (avoid zeros)
            anom = rows[rows['value'] > 0].copy()
        if anom.empty:
            continue
        # group consecutive anomaly rows
        grouped = group_consecutive_anomalies(anom, time_col='timestamp', max_gap=pd.Timedelta(seconds=90))
        if grouped.empty:
            continue
        # attach identifiers and compute percent_over
        grouped['cmdb_id'] = cmdb
        grouped['kpi_name'] = kpi
        grouped['global_p95'] = gp95
        grouped['max_value_in_group'] = grouped['max_value_in_group'].round(6)
        # percent_over: if global_p95==0 => NaN
        grouped['percent_over'] = np.where(
            gp95 > 0,
            ((grouped['max_value_in_group'] - gp95) / gp95 * 100).round(3),
            np.nan
        )
        # reorder columns
        grouped = grouped[[
            'cmdb_id','kpi_name','group_start_timestamp_utc','group_end_timestamp_utc',
            'points_in_group','max_value_in_group','global_p95','percent_over'
        ]]
        groups_list.append(grouped)
    if groups_list:
        groups_df = pd.concat(groups_list, ignore_index=True, sort=False)
        # sort by percent_over desc (NaNs last) then points_in_group desc
        groups_df['percent_sort'] = groups_df['percent_over'].fillna(-np.inf)  # NaN become -inf so they go to bottom
        groups_df = groups_df.sort_values(['percent_sort','points_in_group'], ascending=[False, False]).drop(columns=['percent_sort']).reset_index(drop=True)
    else:
        groups_df = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','group_start_timestamp_utc','group_end_timestamp_utc',
            'points_in_group','max_value_in_group','global_p95','percent_over'
        ])

# Keep output compact: show top 20 rows
groups_df.head(20)

```
Out[3]:
```
Summary of fault groups found in the incident window (2024-01-24 07:53:02 → 2024-01-24 08:23:02 UTC).
Results use each KPI's global P95 computed from the full series, then grouped consecutive anomaly points in the window.

Top groups (sorted by percent_over desc):
1. ts-travel-service — mem: NOT present (no grouped mem anomalies above P95)
2. ts-travel-service — cpu
   - window: 2024-01-24 07:58:00 (single point)
   - points: 1, max = 7.182736
   - global P95 = 4.091429
   - percent_over ≈ 75.56% (largest relative CPU spike)
3. ts-order-service — cpu
   - window: 2024-01-24 07:59:00 (single point)
   - points: 1, max = 5.833515
   - global P95 = 4.421037
   - percent_over ≈ 31.95%
4. ts-travel-service — workload
   - window: 2024-01-24 08:19:00 → 08:20:00 (2 consecutive minutes)
   - points: 2, max = 12.71133
   - global P95 = 10.59760
   - percent_over ≈ 19.95%
5. ts-route-service — socket
   - window: 2024-01-24 08:20:00 (single point)
   - points: 1, max = 22.0
   - global P95 = 19.34
   - percent_over ≈ 13.75%
6. ts-travel-service — socket
   - window: 2024-01-24 08:12:00 (single point)
   - points: 1, max = 21.06667
   - global P95 = 19.49333
   - percent_over ≈ 8.07%
7. ts-train-service — workload
   - window: 2024-01-24 08:19:00 → 08:20:00 (2 points)
   - points: 2, max = 24.02233
   - global P95 = 22.28449
   - percent_over ≈ 7.80%
8. ts-travel-service — log.row_count
   - window: 2024-01-24 08:19:00 (single point)
   - points: 1, max = 1966
   - global P95 = 1847.2
   - percent_over ≈ 6.43%
9. ts-route-service — diskio
   - window: 2024-01-24 08:13:00 (single point)
   - points: 1, max = 45,334,480
   - global P95 = 43,586,520
   - percent_over ≈ 4.01%
10. ts-auth-service — diskio
    - window: 2024-01-24 08:20:00 (single point)
    - points: 1, max ≈ 1,810,451
    - global P95 ≈ 1,721,153
    - percent_over ≈ 5.19%

Other notable single-point groups (smaller percent_over but present): ts-auth cpu (2 points, ~5.12% over), ts-order/ts-train socket and diskio single-point bumps, and log.row_count spikes for ts-auth and ts-order.

Observations and recommended focus
- The largest relative spike is ts-travel-service CPU (75% over P95) — investigate ts-travel-service CPU usage at ~07:58 UTC.
- ts-order-service and ts-travel-service show notable CPU/workload spikes and also elevated log.row_count (traffic/throughput) — these warrant priority investigation.
- Disk I/O spikes occur for ts-route-service and ts-auth-service (single-point but >P95); investigate I/O activity at those timestamps.
- Many KPI groups are single-minute spikes; a few (workload for travel and train) span 2 consecutive minutes.
- mem KPI did not produce grouped anomalies in this output for these services (no mem groups above P95 present here).

Suggested next steps
- Inspect application and system-level metrics (process list, GC, thread counts) on ts-travel-service and ts-order-service around the listed timestamps.
- Check increased log traffic entries for ts-travel, ts-order, and ts-auth at their anomaly timestamps for correlated errors or traffic bursts.
- Review disk I/O sources (backups, heavy queries, or GC-related I/O) for ts-route and ts-auth at the indicated minutes.

If you want, I can extract the raw anomaly rows for any listed group (timestamps and values) for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id       kpi_name group_start_timestamp_utc   group_end_timestamp_utc  points_in_group  max_value_in_group    global_p95  percent_over
0   ts-travel-service            cpu 2024-01-24 07:58:00+00:00 2024-01-24 07:58:00+00:00                1        7.182736e+00  4.091429e+00        75.556
1    ts-order-service            cpu 2024-01-24 07:59:00+00:00 2024-01-24 07:59:00+00:00                1        5.833515e+00  4.421037e+00        31.949
2   ts-travel-service       workload 2024-01-24 08:19:00+00:00 2024-01-24 08:20:00+00:00                2        1.271133e+01  1.059760e+01        19.945
3    ts-route-service         socket 2024-01-24 08:20:00+00:00 2024-01-24 08:20:00+00:00                1        2.200000e+01  1.934000e+01        13.754
4   ts-travel-service         socket 2024-01-24 08:12:00+00:00 2024-01-24 08:12:00+00:00                1        2.106667e+01  1.949333e+01         8.071
5    ts-train-service       workload 2024-01-24 08:19:00+00:00 2024-01-24 08:20:00+00:00                2        2.402233e+01  2.228449e+01         7.798
6    ts-train-service         socket 2024-01-24 08:18:00+00:00 2024-01-24 08:18:00+00:00                1        1.683333e+01  1.570333e+01         7.196
7    ts-order-service       workload 2024-01-24 08:20:00+00:00 2024-01-24 08:20:00+00:00                1        7.488667e+00  7.021013e+00         6.661
8   ts-travel-service  log.row_count 2024-01-24 08:19:00+00:00 2024-01-24 08:19:00+00:00                1        1.966000e+03  1.847200e+03         6.431
9   ts-travel-service         diskio 2024-01-24 08:16:00+00:00 2024-01-24 08:16:00+00:00                1        6.095932e+04  5.738472e+04         6.229
10   ts-order-service  log.row_count 2024-01-24 08:19:00+00:00 2024-01-24 08:19:00+00:00                1        4.460000e+02  4.236000e+02         5.288
11    ts-auth-service         diskio 2024-01-24 08:20:00+00:00 2024-01-24 08:20:00+00:00                1        1.810451e+06  1.721153e+06         5.188
12    ts-auth-service            cpu 2024-01-24 08:02:00+00:00 2024-01-24 08:03:00+00:00                2        1.773783e+01  1.687433e+01         5.117
13   ts-route-service         diskio 2024-01-24 08:13:00+00:00 2024-01-24 08:13:00+00:00                1        4.533448e+07  4.358652e+07         4.010
14    ts-auth-service       workload 2024-01-24 07:56:00+00:00 2024-01-24 07:56:00+00:00                1        2.980362e+00  2.867360e+00         3.941
15    ts-auth-service         socket 2024-01-24 08:07:00+00:00 2024-01-24 08:07:00+00:00                1        1.470000e+01  1.424333e+01         3.206
16   ts-order-service         diskio 2024-01-24 08:15:00+00:00 2024-01-24 08:15:00+00:00                1        1.878737e+06  1.821038e+06         3.168
17    ts-auth-service  log.row_count 2024-01-24 07:57:00+00:00 2024-01-24 07:57:00+00:00                1        3.500000e+02  3.400000e+02         2.941
18   ts-order-service            cpu 2024-01-24 08:14:00+00:00 2024-01-24 08:14:00+00:00                1        4.542431e+00  4.421037e+00         2.746
19   ts-route-service       workload 2024-01-24 08:19:00+00:00 2024-01-24 08:20:00+00:00                2        3.622233e+01  3.527090e+01         2.698```
```

In [ ]:
# In[4]:

# Select trace rows with trace_name containing 'travel-service', compute global P95 per (cmdb_id, trace_name),
# then find anomalies in the incident window where value >= global_p95 and summarize compactly.

import pandas as pd
import numpy as np

# Reuse existing 'trace' DataFrame (timestamps already parsed as UTC earlier)
# Incident window (UTC)
start = pd.to_datetime('2024-01-24 07:53:02', utc=True)
end   = pd.to_datetime('2024-01-24 08:23:02', utc=True)

# Filter trace names containing 'travel-service'
mask = trace['trace_name'].str.contains('travel-service', na=False)
trace_travel = trace[mask].copy()

# If no matching rows, return empty table
if trace_travel.empty:
    travel_trace_anomalies = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count_in_window',
                                                  'earliest_anomaly_timestamp_utc','max_value_in_window','global_p95'])
else:
    # Compute global P95 per (cmdb_id, trace_name) using full available series (not windowed)
    p95_df = trace_travel.groupby(['cmdb_id','trace_name'])['value'].quantile(0.95).reset_index(name='global_p95')

    # Filter trace_travel to incident window
    trace_window = trace_travel[trace_travel['timestamp'].between(start, end)].copy()
    if trace_window.empty:
        travel_trace_anomalies = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count_in_window',
                                                      'earliest_anomaly_timestamp_utc','max_value_in_window','global_p95'])
    else:
        # Merge window rows with global_p95
        merged = trace_window.merge(p95_df, on=['cmdb_id','trace_name'], how='left')
        # Keep rows where value >= global_p95 (handle possible NaNs: drop if global_p95 is null)
        merged = merged.dropna(subset=['global_p95'])
        anomalies = merged[merged['value'] >= merged['global_p95']].copy()
        if anomalies.empty:
            travel_trace_anomalies = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count_in_window',
                                                          'earliest_anomaly_timestamp_utc','max_value_in_window','global_p95'])
        else:
            # Aggregate per group
            agg = anomalies.groupby(['cmdb_id','trace_name']).agg(
                anomaly_count_in_window = ('value','count'),
                earliest_anomaly_timestamp_utc = ('timestamp','min'),
                max_value_in_window = ('value','max'),
                global_p95 = ('global_p95','first')
            ).reset_index()
            # Round numeric columns for compactness
            agg['max_value_in_window'] = agg['max_value_in_window'].round(6)
            agg['global_p95'] = agg['global_p95'].round(6)
            # Sort as requested
            travel_trace_anomalies = agg.sort_values(['anomaly_count_in_window','max_value_in_window'], ascending=[False, False]).reset_index(drop=True)
            # Keep only required columns (already)
# Show compact top 20
travel_trace_anomalies.head(20)

```
Out[4]:
```
Summary of trace anomalies for traces containing "travel-service" (incident window 2024-01-24 07:53:02 → 2024-01-24 08:23:02 UTC):

- Many trace error_rate series across services (e.g., root, ts-order-service, ts-route-service, ts-seat-service, ts-travel-service, ts-train-service, etc.) show global P95 = 0 and appear flagged in the window (anomaly_count = 24–25, max = 0). These are not actionable (all zeros) and can be ignored.

- Actionable non-zero anomalies (sorted by relevance):
  1. ts-seat-service — trace.from_ts-travel-service.row_count
     - anomaly_count_in_window: 3
     - earliest_anomaly_timestamp_utc: 2024-01-24 08:02:00+00:00
     - max_value_in_window: 116
     - global_p95: 112
  2. ts-order-service — trace.from_ts-travel-service.row_count
     - anomaly_count_in_window: 3
     - earliest_anomaly_timestamp_utc: 2024-01-24 08:02:00+00:00
     - max_value_in_window: 58
     - global_p95: 56
  3. ts-preserve-service — trace.to_ts-travel-service.row_count
     - anomaly_count_in_window: 3
     - earliest_anomaly_timestamp_utc: 2024-01-24 08:06:00+00:00
     - max_value_in_window: 8
     - global_p95: 7

Interpretation and recommended next steps:
- Ignore the zero-valued error_rate flags (P95=0). Focus on the row_count spikes around ~08:02 UTC for ts-seat-service and ts-order-service (and a smaller spike for ts-preserve-service at ~08:06). These indicate increased request/row traffic involving travel-service flows and are likely the most relevant trace signals for this incident.
- Investigate request patterns, upstream callers, and logs at those timestamps (08:02–08:06 UTC) for ts-seat-service and ts-order-service to see whether traffic surge or downstream slowness/errors caused the increased row counts.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                     trace_name  anomaly_count_in_window earliest_anomaly_timestamp_utc  max_value_in_window  global_p95
0                      root          trace.to_ts-travel-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
1          ts-order-service        trace.from_ts-travel-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
2          ts-route-service        trace.from_ts-travel-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
3           ts-seat-service        trace.from_ts-travel-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
4           ts-seat-service          trace.to_ts-travel-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
5     ts-ticketinfo-service        trace.from_ts-travel-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
6          ts-train-service        trace.from_ts-travel-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
7         ts-travel-service        trace.from_ts-travel-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
8         ts-travel-service          trace.to_ts-travel-service.error_rate                       25      2024-01-24 07:56:00+00:00                  0.0         0.0
9                      root    trace.to_ts-admin-travel-service.error_rate                       24      2024-01-24 07:56:00+00:00                  0.0         0.0
10  ts-admin-travel-service  trace.from_ts-admin-travel-service.error_rate                       24      2024-01-24 07:56:00+00:00                  0.0         0.0
11  ts-admin-travel-service    trace.to_ts-admin-travel-service.error_rate                       24      2024-01-24 07:56:00+00:00                  0.0         0.0
12  ts-admin-travel-service          trace.to_ts-travel-service.error_rate                       24      2024-01-24 07:56:00+00:00                  0.0         0.0
13          ts-food-service          trace.to_ts-travel-service.error_rate                       24      2024-01-24 07:56:00+00:00                  0.0         0.0
14      ts-preserve-service          trace.to_ts-travel-service.error_rate                       24      2024-01-24 07:56:00+00:00                  0.0         0.0
15        ts-travel-service  trace.from_ts-admin-travel-service.error_rate                       24      2024-01-24 07:56:00+00:00                  0.0         0.0
16       ts-travel2-service  trace.from_ts-admin-travel-service.error_rate                       24      2024-01-24 07:56:00+00:00                  0.0         0.0
17          ts-seat-service         trace.from_ts-travel-service.row_count                        3      2024-01-24 08:02:00+00:00                116.0       112.0
18         ts-order-service         trace.from_ts-travel-service.row_count                        3      2024-01-24 08:02:00+00:00                 58.0        56.0
19      ts-preserve-service           trace.to_ts-travel-service.row_count                        3      2024-01-24 08:06:00+00:00                  8.0         7.0```
```